# USD - DM affine SV calibration — per-period caplets

Turnkey calibration notebook. **Requires torch + a GPU.**

**Run order:** Config → Setup → **1. Shakeout** (fast, ~1–2 min — confirms the wiring runs) → **2. Full run** (the nested-model ladder, GPU, walk away) → **3. Results & figures**.

Outputs land in `dm_calibration/runs/` (checkpoints `.pt`, per-caplet `results_*.csv`, `ladder_*.json`) and `dm_calibration/figures/`.

The engine is `dm_calibrate.py` (per-period pricer + vol-RMSE + Adam-SPSA, reusing the tested `scripts/` simulator). First run may surface torch-wiring bugs — that's what the shakeout is for.

In [ ]:
# ==================== CONFIG - edit these, then Run All ====================
CCY     = 'usd'         # eur | usd | gbp
MODELS  = ['HWH']       # just the stochastic-vol model (the nested ladder is a separate run)
ITERS   = 500           # SPSA steps
PATHS   = 8192          # MC paths / eval (doubled by antithetic)
MAX_MAT = 10.0          # cap caplet maturity (yrs)
DT      = 0.004         # MC time step (yrs). 0.004 = DAILY; 0.02 = weekly (gives 9.4bp, ~5x faster)
# Result saves to runs/results_{CCY}_HWH_qe.csv ; view it in {CCY}_results.ipynb.
# NOTE (daily on a display GPU): long runs can hit Windows TDR and freeze ~iter 100.
#   Fix: regedit HKLM\SYSTEM\CurrentControlSet\Control\GraphicsDrivers ->
#        new DWORD TdrDelay = 60 -> reboot.  Or just use weekly DT=0.02.
#   Close other GPU apps (browser/Bloomberg/other kernels) before running.
# ==========================================================================

In [ ]:
# ---- setup: locate dm_calibration/ and import the engine ----
import os, sys, json, importlib
cand = os.path.abspath("")
found = None
for _ in range(6):
    if os.path.exists(os.path.join(cand, "dm_calibrate.py")):
        found = cand; break
    sub = os.path.join(cand, "dm_calibration")
    if os.path.exists(os.path.join(sub, "dm_calibrate.py")):
        found = sub; break
    cand = os.path.dirname(cand)
assert found, "Run this notebook from within the repo (couldn't find dm_calibrate.py)"
sys.path.insert(0, found)
import dm_calibrate as dc; importlib.reload(dc)
import dm_figures as dfig; importlib.reload(dfig)
import torch, pandas as pd
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("dm_calibration:", found)
print("device:", device, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# ---- data sanity: the stripped target surface must exist ----
cap = dc.load_market(CCY, device)
print(f"{CCY.upper()}: {len(cap)} caplets, tau={dc.TENOR[CCY]}, "
      f"maturities {cap.t_fix.min():.2f}–{cap.t_fix.max():.1f}Y, "
      f"strikes {cap.strike.min()*100:.1f}–{cap.strike.max()*100:.1f}%")
cap.head()

## 1. Shakeout — verify the torch wiring (fast)
Tiny settings so it finishes in ~1–2 min. If this runs without error and prints a loss, the pipeline works; proceed to the full run. If it errors, it's a torch-wiring bug to fix (the per-period math itself is validated in `dm_pricer.py`).

In [ ]:
_ = dc.calibrate(CCY, "HWH", device, max_iter=40, n_paths=4096, max_mat=MAX_MAT, dt=DT)

## 2. Full calibration — the nested-model ladder
Production settings from CONFIG. With `MODELS = ["1F_Gauss","2F_Gauss","HW_detvol","HWH"]` this is Roland's rec-2 ladder. Safe to run and walk away.

In [ ]:
results = [dc.calibrate(CCY, m, device, max_iter=ITERS, n_paths=PATHS,
                        max_mat=MAX_MAT, dt=DT, sim='qe') for m in MODELS]
print('
===== ' + CCY.upper() + ' =====')
for r in results:
    print(f"{r['model']:<10} vol-RMSE = {r['rmse_bp']} bp")

## 3. Results & figures
Market surface + smile need no results. The market-vs-model overlay and the relative/bucketed/ATM-OTM metrics (Roland rec 4) use the HWH per-caplet results CSV.

In [ ]:
print("market:", dfig.fig_market_surface(CCY))
print("smile :", dfig.fig_smile(CCY))
res_csv = os.path.join(dc.RUNS, f"results_{CCY}_HWH.csv")
if os.path.exists(res_csv):
    res = pd.read_csv(res_csv)
    print("overlay:", dfig.fig_overlay(res, CCY))
    print("\nerror metrics (rec 4):")
    print(json.dumps(dfig.error_metrics(res), indent=2))
else:
    print("(run the HWH calibration first to produce", os.path.basename(res_csv), ")")

---
**Outputs:** `runs/calibration_weights/{ccy}_{model}.pt` (params), `runs/results_{ccy}_{model}.csv` (per-caplet market vs model), `runs/ladder_{ccy}.json` (RMSE ladder), `figures/fig_{ccy}_*.png`.

Send me `ladder_{ccy}.json` + `results_{ccy}_HWH.csv` and I'll fill the article's §5 and finalize the figures. Repeat with `CCY="gbp"` / `"usd"` for the robustness section.